# PANDA — Final submission notebook

Runs the ordinal + MSE ensemble on the test set and produces `submission.csv`.

**Attach on Kaggle:**
1. `prostate-cancer-grade-assessment` (official PANDA competition dataset)
2. `panda-effnetb0-ordinal-5fold` (your private weights)
3. `panda-effnetb0-mse-5fold` (your private weights)
4. `panda-src` (your private src code dataset)

**Settings:** GPU T4 ON, Internet OFF


In [ ]:
BATCH_SIZE = 16
ORDINAL_MODE = 'threshold'
N_FOLDS = 5
OUTPUT_CSV = '/kaggle/working/submission.csv'

MODEL_FAMILIES = [
    {
        'name': 'b0_ordinal',
        'weights_dir': '/kaggle/input/datasets/gojamodicsila/panda-effnetb0-ordinal-5fold',
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_ordinal_fold{fold}.pth',
        'model_kind': 'baseline',
    },
    {
        'name': 'b0_mse',
        'weights_dir': '/kaggle/input/datasets/gojamodicsila/panda-effnetb0-mse-5fold',
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_mse_fold{fold}.pth',
        'model_kind': 'baseline',
    },
]


In [ ]:
# Install efficientnet (no internet needed, it's in the base image)
!pip install -q efficientnet_pytorch


In [ ]:
import glob
import os
import sys
import zipfile

# Load src from dataset instead of GitHub
src_zip_candidates = glob.glob('/kaggle/input/**/src.zip', recursive=True)
if src_zip_candidates:
    with zipfile.ZipFile(src_zip_candidates[0], 'r') as z:
        z.extractall('/kaggle/working/repo')
    print('Extracted src from:', src_zip_candidates[0])
else:
    raise FileNotFoundError('Could not find src.zip. Attach panda-src dataset.')

sys.path.insert(0, '/kaggle/working/repo')

import numpy as np
import pandas as pd
import torch

from src.dataset import PandaDataset
from src.eval import round_preds
from src.inference import load_model, predict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# Find test images
candidates = glob.glob('/kaggle/input/**/test_images', recursive=True)
TEST_DIR = next((c for c in candidates if os.path.isdir(c)), None)
if TEST_DIR is None:
    raise RuntimeError(f'Could not find test images. Input: {os.listdir("/kaggle/input")}')
print('TEST_DIR:', TEST_DIR)
print('Test files:', len(os.listdir(TEST_DIR)))


In [ ]:
# Load test CSV
test_csv = glob.glob('/kaggle/input/**/test.csv', recursive=True)
if not test_csv:
    raise RuntimeError('Could not find test.csv')
test_df = pd.read_csv(test_csv[0])
print('Test slides:', len(test_df))
print(test_df.head())

# Add dummy isup_grade for dataset compatibility
test_df['isup_grade'] = 0


In [ ]:
# Run ensemble inference
test_dataset = PandaDataset(test_df, TEST_DIR, train=False)
all_family_preds = []

for family in MODEL_FAMILIES:
    fold_preds = []
    for fold in range(N_FOLDS):
        weight_path = os.path.join(
            family['weights_dir'],
            family['weight_pattern'].format(fold=fold)
        )
        if not os.path.exists(weight_path):
            raise FileNotFoundError(f'Missing: {weight_path}')
        model = load_model(
            weight_path,
            backbone=family.get('backbone', 'efficientnet-b0'),
            device=device,
            model_kind=family.get('model_kind', 'baseline'),
        )
        preds = predict(
            model, test_dataset, device,
            batch_size=BATCH_SIZE,
            ordinal_mode=ORDINAL_MODE,
        )
        fold_preds.append(preds)
        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        print(f"{family['name']} fold {fold} done")
    all_family_preds.append(np.mean(fold_preds, axis=0))

ensemble_preds = np.mean(all_family_preds, axis=0)
final_preds = round_preds(ensemble_preds)
print('Predictions done. Grade distribution:', np.bincount(final_preds))


In [ ]:
# Save submission
submission = pd.DataFrame({
    'image_id': test_df.image_id.values,
    'isup_grade': final_preds,
})
submission.to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV)
print(submission.head(10))
print('Grade distribution:')
print(submission.isup_grade.value_counts().sort_index())
